# ThermoNO vs the classical layered backbone (report Sec 9.26), multi-seed, geometry4/5/6

**What this tests.** On CPU (geometry4, seed 0, 5-fold CV) ThermoNO beat the training-free
layered backbone on field R2 (0.995 vs 0.981), hotspot localisation (1000 vs 3536 um) and
peak error (0.78 vs 2.06 K), but lost on top-1% recall (0.70 vs 0.81). This notebook asks
whether that holds (a) across seeds, which change both the folds and the initialisation, and
(b) on geometry5 and geometry6, where the stack is thicker and more heterogeneous.

**What it runs.** `scripts/thermono_train.py` from the cloned repo, unchanged, on the GPU.
Every geometry x seed is a full 5-fold CV. Early stopping uses a hold-out drawn from the
training fold, never the test fold. The backbone is scored on the same test scenarios, so each
row is a paired comparison.

**Setup.**
1. Attach the v5 shelf datasets you have: `geometry4_shelf_v5.zip`, `geometry5_shelf_v5.zip`,
   `geometry6_shelf_v5.zip` (from `data/kaggle_v5/`). Slugs do not matter; files are found by
   name. Missing geometries are skipped.
2. Settings -> Internet -> On (the repo is cloned). Accelerator -> GPU T4 x1.
3. Run all. Download `/kaggle/working/thermono_results/` when it finishes.

**Expected time.** The backbone cache takes about 30 s per geometry (CPU). Training is a few
minutes per geometry x seed on a T4; geometry6 is the slowest.

In [ ]:
import subprocess, sys
import torch
print('Python :', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Enable Settings -> Accelerator -> GPU T4 x1 and restart the session.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'pyamg'], check=True)

In [ ]:
import shutil, subprocess, sys
from pathlib import Path

REPO = 'https://github.com/rajul-kk/thermo-3dic-surrogates.git'
INPUT = Path('/kaggle/input')
CLONE = Path('/kaggle/working/repo')
DATA = Path('/kaggle/working/data')          # data/3d-ice-layout-<geom>/<geom>/*.npz, as in the repo
OUT = Path('/kaggle/working/thermono_results')
OUT.mkdir(parents=True, exist_ok=True)

# Always sync to the latest main: /kaggle/working persists across sessions.
shutil.rmtree(CLONE, ignore_errors=True)
r = subprocess.run(['git', 'clone', '--depth', '1', REPO, str(CLONE)], capture_output=True, text=True)
assert r.returncode == 0, 'git clone failed; turn Settings -> Internet -> On.\n' + r.stderr
sha = subprocess.run(['git', '-C', str(CLONE), 'rev-parse', '--short', 'HEAD'],
                     capture_output=True, text=True).stdout.strip()
print('repo cloned at commit', sha)

GEOMS = []
for geom in ['geometry4', 'geometry5', 'geometry6']:
    files = sorted({p.name: p for p in INPUT.rglob(geom + '_*.npz')}.values(), key=lambda p: p.name)
    if not files:
        print(f'{geom}: not attached, skipped')
        continue
    dst = DATA / f'3d-ice-layout-{geom}' / geom
    shutil.rmtree(dst, ignore_errors=True)
    dst.mkdir(parents=True)
    for p in files:
        (dst / p.name).symlink_to(p)
    flag = '' if len(files) == 45 else '   <-- expected 45!'
    print(f'{geom}: {len(files)} scenarios{flag}')
    GEOMS.append(geom)
assert GEOMS, 'No geometry4/5/6 .npz attached. Upload data/kaggle_v5/geometry*_shelf_v5.zip as datasets.'

In [ ]:
# Same defaults as the CPU run in report Sec 9.26; only the seed and geometry vary.
SEEDS = [0, 1, 2]
EXTRA = []                      # e.g. ['--ch', '32'] for a wider model; leave empty to match the report
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('geometries', GEOMS, '| seeds', SEEDS, '| device', DEVICE)

In [ ]:
import json, time

for geom in GEOMS:
    for seed in SEEDS:
        res = CLONE / 'results' / f'thermono_{geom}_seed{seed}.json'
        if res.exists():
            print(f'{geom} seed {seed}: already done')
            continue
        print(f'=== {geom} seed {seed}', flush=True)
        t = time.time()
        cmd = [sys.executable, 'scripts/thermono_train.py', geom, '--seed', str(seed),
               '--device', DEVICE, '--data-root', str(DATA)] + EXTRA
        p = subprocess.Popen(cmd, cwd=CLONE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print('  ' + line, end='', flush=True)
        assert p.wait() == 0, f'{geom} seed {seed} failed'
        shutil.copy(res, OUT / res.name)
        print(f'  ({(time.time() - t) / 60:.1f} min)')

In [ ]:
import numpy as np

KEYS = [('r2_mean', 'R2 mean', '{:.4f}'), ('r2_median', 'R2 median', '{:.4f}'),
        ('det_mae_K', 'det.MAE K', '{:.3f}'), ('loc_err_um_median', 'loc um', '{:.0f}'),
        ('recall_mean', 'recall', '{:.3f}'), ('abs_peak_err_K', '|peak| K', '{:.2f}')]
BETTER_HIGH = {'r2_mean', 'r2_median', 'recall_mean'}

table = {}
for geom in GEOMS:
    runs = [json.loads((OUT / f'thermono_{geom}_seed{s}.json').read_text())['summary']
            for s in SEEDS if (OUT / f'thermono_{geom}_seed{s}.json').exists()]
    if not runs:
        continue
    print(f'\n{geom}: {len(runs)} seed(s), mean +- std across seeds')
    print(f"{'model':<10}" + ''.join(f'{lab:>18}' for _, lab, _ in KEYS))
    for model in ('backbone', 'thermono'):
        cells = []
        for k, _, fmt in KEYS:
            v = np.array([r[model][k] for r in runs])
            cells.append(f'{fmt.format(v.mean())} +- {fmt.format(v.std())}')
        print(f'{model:<10}' + ''.join(f'{c:>18}' for c in cells))
    wins = []
    for k, lab, _ in KEYS:
        d = np.array([r['thermono'][k] - r['backbone'][k] for r in runs])
        good = d > 0 if k in BETTER_HIGH else d < 0
        wins.append(f'{lab}: {int(good.sum())}/{len(runs)}')
    print('ThermoNO better than backbone in seeds ->', ', '.join(wins))
    table[geom] = runs

(OUT / 'summary.json').write_text(json.dumps(table, indent=1))
print('\nCPU reference (report Sec 9.26, geometry4 seed 0): backbone R2 0.981, loc 3536 um, '
      'recall 0.810, |peak| 2.06 K; ThermoNO R2 0.995, loc 1000 um, recall 0.700, |peak| 0.78 K.')
print('Download', OUT, 'and send back summary.json plus the per-seed JSONs.')

## Reading the result

- **The claim survives** if ThermoNO beats the backbone on R2, localisation and peak error in
  most seeds on every attached geometry.
- **Recall is the known weak spot.** If it stays below the backbone everywhere, the report
  states that ThermoNO trades hot-region shape for peak placement.
- geometry4 has diffuse peaks (report Sec 9.17), so its localisation distance is descriptive
  only; geometry5/6 localisation carries more weight.